![Tree](https://i.ytimg.com/vi/xiu0nqxeOic/hq720.jpg?sqp=-oaymwEhCK4FEIIDSFryq4qpAxMIARUAAAAAGAElAADIQj0AgKJD&rs=AOn4CLDtttZQr8-G9NnMyj5itrdI-5cvKg)

# Santa 2025 — Christmas Tree Packing Challenge 🧩🎄  
### Fast bbox3 sweep + safe per-group rotation polish (improved baseline)

This notebook is based on **saspav’s baseline**:  
https://www.kaggle.com/code/saspav/santa-submission

I kept the original idea (use `bbox3` as a strong heuristic packer) and focused on making the pipeline **more stable, safer and more optimizable** under the competition metric.

✅ **Best public score achieved with this notebook:** **70.628146846969**  
(lower is better: we minimize ∑ S(n)² / n)

---

## What I changed vs the original notebook

### 1) “Best-so-far” loop (no regression)
The baseline workflow could overwrite a good submission with a worse one during parameter sweeps.  
I implemented a **best checkpoint** strategy:
- always start each run from the current best submission  
- accept a new candidate only if it improves the score  
- rollback otherwise

This makes optimization stable and reproducible.

### 2) Fast & reliable scoring (bounds-based)
Scoring is computed using **min/max polygon bounds** instead of `unary_union()` for each group.  
It is significantly faster and avoids heavy geometry ops.

### 3) Safe overlap validation (STRtree)
Overlap checks are accelerated using **Shapely STRtree**, so we can validate candidates efficiently:
- quick score mode (no overlap check)  
- full validation only for candidates that look promising

### 4) Per-group rotation polish (local improvement)
A key improvement: **each N-group can be rotated independently** (metric is independent per group).  
I added a `fix_direction()` step that:
- estimates an optimal global rotation angle using convex hull
- applies the rotation
- **accepts it only if the REAL side length improves after rotation**

This prevents “false improvements” due to hull approximation / numerical drift.

In [ ]:
DEBUG = True

MAX_HOURS = 1.7

In [ ]:
from shutil import copy

copy('/kaggle/input/santa-2025-csv/santa-2025.csv', '/kaggle/working/submission.csv')
copy('/kaggle/input/santa-2025-csv/bbox3', '/kaggle/working/')

In [ ]:
!chmod +x ./bbox3

In [ ]:
import numpy as np
import pandas as pd
from decimal import Decimal, getcontext
from shapely import affinity, touches
from shapely.geometry import Polygon
from shapely.ops import unary_union
from scipy.spatial import ConvexHull
from scipy.optimize import minimize_scalar

getcontext().prec = 30
scale_factor = 1


class ChristmasTree:
    """Represents a single, rotatable Christmas tree of a fixed size."""

    def __init__(self, center_x='0', center_y='0', angle='0'):
        """Initializes the Christmas tree with a specific position and rotation."""
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)

        trunk_w = Decimal('0.15')
        trunk_h = Decimal('0.2')
        base_w = Decimal('0.7')
        mid_w = Decimal('0.4')
        top_w = Decimal('0.25')
        tip_y = Decimal('0.8')
        tier_1_y = Decimal('0.5')
        tier_2_y = Decimal('0.25')
        base_y = Decimal('0.0')
        trunk_bottom_y = -trunk_h

        initial_polygon = Polygon(
            [
                # Start at Tip
                (Decimal('0.0') * scale_factor, tip_y * scale_factor),
                # Right side - Top Tier
                (top_w / Decimal('2') * scale_factor, tier_1_y * scale_factor),
                (top_w / Decimal('4') * scale_factor, tier_1_y * scale_factor),
                # Right side - Middle Tier
                (mid_w / Decimal('2') * scale_factor, tier_2_y * scale_factor),
                (mid_w / Decimal('4') * scale_factor, tier_2_y * scale_factor),
                # Right side - Bottom Tier
                (base_w / Decimal('2') * scale_factor, base_y * scale_factor),
                # Right Trunk
                (trunk_w / Decimal('2') * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal('2') * scale_factor, trunk_bottom_y * scale_factor),
                # Left Trunk
                (-(trunk_w / Decimal('2')) * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal('2')) * scale_factor, base_y * scale_factor),
                # Left side - Bottom Tier
                (-(base_w / Decimal('2')) * scale_factor, base_y * scale_factor),
                # Left side - Middle Tier
                (-(mid_w / Decimal('4')) * scale_factor, tier_2_y * scale_factor),
                (-(mid_w / Decimal('2')) * scale_factor, tier_2_y * scale_factor),
                # Left side - Top Tier
                (-(top_w / Decimal('4')) * scale_factor, tier_1_y * scale_factor),
                (-(top_w / Decimal('2')) * scale_factor, tier_1_y * scale_factor),
            ]
        )
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(rotated,
                                          xoff=float(self.center_x * scale_factor),
                                          yoff=float(self.center_y * scale_factor))

    def clone(self) -> "ChristmasTree":
        return ChristmasTree(
            center_x=str(self.center_x),
            center_y=str(self.center_y),
            angle=str(self.angle),
        )


def get_tree_list_side_lenght(tree_list: list[ChristmasTree]) -> Decimal:
    # быстрее и точнее для нашей цели, чем unary_union().bounds
    if not tree_list:
        return Decimal("0")

    xmin = Decimal("1e30")
    ymin = Decimal("1e30")
    xmax = Decimal("-1e30")
    ymax = Decimal("-1e30")

    for t in tree_list:
        x0, y0, x1, y1 = t.polygon.bounds
        if x0 < xmin: xmin = Decimal(str(x0))
        if y0 < ymin: ymin = Decimal(str(y0))
        if x1 > xmax: xmax = Decimal(str(x1))
        if y1 > ymax: ymax = Decimal(str(y1))

    side = max(xmax - xmin, ymax - ymin)
    return side / scale_factor


def get_total_score(dict_of_side_length: dict[str, Decimal]):
    score = 0
    for k, v in dict_of_side_length.items():
        score += v ** 2 / Decimal(k)
    return score


def parse_csv(csv_path) -> dict[str, list[ChristmasTree]]:
    print(f'\nparse_csv: {csv_path=}')

    result = pd.read_csv(csv_path)
    result['x'] = result['x'].str.strip('s')
    result['y'] = result['y'].str.strip('s')
    result['deg'] = result['deg'].str.strip('s')
    result[['group_id', 'item_id']] = result['id'].str.split('_', n=2, expand=True)

    dict_of_tree_list = {}
    dict_of_side_length = {}
    for group_id, group_data in result.groupby('group_id'):
        tree_list = [ChristmasTree(center_x=row['x'], center_y=row['y'], angle=row['deg'])
                     for _, row in group_data.iterrows()]
        dict_of_tree_list[group_id] = tree_list
        dict_of_side_length[group_id] = get_tree_list_side_lenght(tree_list)

    return dict_of_tree_list, dict_of_side_length


def calculate_bbox_side_at_angle(angle_deg, points):
    angle_rad = np.radians(angle_deg)
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    rot_matrix_T = np.array([[c, s], [-s, c]])
    rotated_points = points.dot(rot_matrix_T)
    min_xy = np.min(rotated_points, axis=0);
    max_xy = np.max(rotated_points, axis=0)
    return max(max_xy[0] - min_xy[0], max_xy[1] - min_xy[1])


def optimize_rotation(trees):
    all_points = []
    for tree in trees: all_points.extend(list(tree.polygon.exterior.coords))
    points_np = np.array(all_points)

    hull_points = points_np[ConvexHull(points_np).vertices]

    initial_side = calculate_bbox_side_at_angle(0, hull_points)

    res = minimize_scalar(lambda a: calculate_bbox_side_at_angle(a, hull_points),
                          bounds=(0.001, 89.999), method='bounded')
    found_angle_deg = res.x
    found_side = res.fun

    improvement = initial_side - found_side

    EPSILON = 1e-8

    if improvement > EPSILON:
        best_angle_deg = found_angle_deg
        best_side = Decimal(found_side) / scale_factor
    else:
        best_angle_deg = 0.0
        best_side = Decimal(initial_side) / scale_factor

    return best_side, best_angle_deg


def apply_rotation(trees, angle_deg):
    if not trees or abs(angle_deg) < 1e-9: return [t.clone() for t in trees]

    bounds = [t.polygon.bounds for t in trees]
    min_x = min(b[0] for b in bounds);
    min_y = min(b[1] for b in bounds)
    max_x = max(b[2] for b in bounds);
    max_y = max(b[3] for b in bounds)
    rotation_center = np.array([(min_x + max_x) / 2.0, (min_y + max_y) / 2.0])

    angle_rad = np.radians(angle_deg)
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    rot_matrix = np.array([[c, -s], [s, c]])

    points = np.array([[float(t.center_x), float(t.center_y)] for t in trees])
    shifted = points - rotation_center
    rotated = shifted.dot(rot_matrix.T) + rotation_center

    rotated_trees = []
    for i in range(len(trees)):
        new_tree = ChristmasTree(Decimal(rotated[i, 0]), Decimal(rotated[i, 1]),
                                 Decimal(trees[i].angle + Decimal(angle_deg)))
        rotated_trees.append(new_tree)
    return rotated_trees


def fix_direction(current_solution_path='submission.csv', out_file='submission.csv'):
    dict_of_tree_list, dict_of_side_length = parse_csv(current_solution_path)
    current_score = get_total_score(dict_of_side_length)
    print(f'[fix_direction] current_score={float(current_score):0.12f}')

    EPS = Decimal("1e-10")
    improved = False

    for group_id, trees in dict_of_tree_list.items():
        old_side = dict_of_side_length[group_id]
        _, best_angle_deg = optimize_rotation(trees)

        if abs(best_angle_deg) < 1e-9:
            continue

        rotated = apply_rotation(trees, best_angle_deg)
        new_side_real = get_tree_list_side_lenght(rotated)

        if old_side - new_side_real > EPS:
            ok = True
            for t in rotated:
                if abs(t.center_x) > 100 or abs(t.center_y) > 100:
                    ok = False
                    break
            if ok:
                dict_of_tree_list[group_id] = rotated
                dict_of_side_length[group_id] = new_side_real
                improved = True

    new_score = get_total_score(dict_of_side_length)
    diff = current_score - new_score
    print(f'[fix_direction] new_score={float(new_score):0.12f}  diff={float(diff):0.12f}')

    if improved and diff > Decimal("0"):
        tree_data = []
        for group_name, tree_list in dict_of_tree_list.items():
            for item_id, tree in enumerate(tree_list):
                tree_data.append({
                    'id': f'{group_name}_{item_id}',
                    'x': f's{tree.center_x}',
                    'y': f's{tree.center_y}',
                    'deg': f's{tree.angle}'
                })
        pd.DataFrame(tree_data).to_csv(out_file, index=False)

    return current_score, new_score


In [ ]:
import os
import time
import subprocess
import threading
from shutil import copy
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

def score_and_validate_submission(file_path: str, max_n: int = 200, check_overlaps: bool = True) -> dict:
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        return {"status": "FAILED", "error": "File Not Found"}
    except Exception as e:
        return {"status": "FAILED", "error": f"CSV Read Error: {e}"}

    if check_overlaps:
        from shapely.strtree import STRtree

    df["x"] = df["x"].astype(str).str.strip("s")
    df["y"] = df["y"].astype(str).str.strip("s")
    df["deg"] = df["deg"].astype(str).str.strip("s")
    df[["group_id", "item_id"]] = df["id"].str.split("_", n=2, expand=True)

    total_score = 0.0
    failed_overlap_n = []

    for gid, g in df.groupby("group_id"):
        n = int(gid)
        if n > max_n:
            continue

        trees = [ChristmasTree(row["x"], row["y"], row["deg"]) for _, row in g.iterrows()]

        xmin = 1e100
        ymin = 1e100
        xmax = -1e100
        ymax = -1e100

        polys = []
        bounds = []
        for t in trees:
            p = t.polygon
            polys.append(p)
            b = p.bounds
            bounds.append(b)
            x0, y0, x1, y1 = b
            if x0 < xmin: xmin = x0
            if y0 < ymin: ymin = y0
            if x1 > xmax: xmax = x1
            if y1 > ymax: ymax = y1

        side = max(xmax - xmin, ymax - ymin)
        total_score += (side * side) / float(n)

        if check_overlaps:
            tree = STRtree(polys)

            def _as_indices(q):
                if len(q) == 0:
                    return []
                first = q[0]
                if isinstance(first, (int, np.integer)):
                    return q
                id_map = {id(geom): i for i, geom in enumerate(polys)}
                return [id_map.get(id(geom), -1) for geom in q]

            overlapped = False
            for i, p in enumerate(polys):
                cand = tree.query(p)
                idxs = _as_indices(cand)
                b1 = bounds[i]
                for j in idxs:
                    if j <= i or j < 0:
                        continue
                    b2 = bounds[j]
                    if (b1[2] < b2[0] or b2[2] < b1[0] or b1[3] < b2[1] or b2[3] < b1[1]):
                        continue
                    pj = polys[j]
                    if p.intersects(pj) and not p.touches(pj):
                        overlapped = True
                        break
                if overlapped:
                    break

            if overlapped:
                failed_overlap_n.append(n)

    status = "SUCCESS" if not failed_overlap_n else "FAILED (Overlaps)"
    return {"status": status, "total_score": float(total_score), "failed_overlap_n": failed_overlap_n}


def run_bbox_simple_with_timeout(debug=False):
    import random
    import os
    import subprocess
    from shutil import copy
    from datetime import datetime, timedelta

    os.makedirs("bbox_sub", exist_ok=True)
    log_file = "bbox_experiments.log"

    start_time = datetime.now()
    timeout = timedelta(hours=MAX_HOURS)

    best_path = "best_submission.csv"

    _, _ = fix_direction("submission.csv", "submission.csv")
    base = score_and_validate_submission("submission.csv", max_n=200, check_overlaps=True)
    if base["status"].startswith("FAILED"):
        raise RuntimeError(f"Initial submission invalid: {base}")

    best_score = base["total_score"]
    copy("submission.csv", best_path)

    if debug:
        candidates = [
            (600, 80),
            (1000, 100),
            (1500, 160),
            (2000, 200),
            (2000, 160),
            (1500, 120),
            (1200, 120),
            (800, 100),
        ]
    else:
        n_grid_fast = [400, 600, 800, 1000]
        r_grid_fast = [60, 80, 100]
        n_grid_heavy = [1200, 1500, 1800, 2000]
        r_grid_heavy = [120, 160, 200]
        candidates = [(n, r) for r in r_grid_fast for n in n_grid_fast]
        random.shuffle(candidates)
        candidates += [(n, r) for r in r_grid_heavy for n in n_grid_heavy]

    completed_runs = 0

    print(f"Start: {start_time}  MAX_HOURS={MAX_HOURS}")
    print(f"Initial best_score={best_score:.12f}")

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*60}\nSTART {start_time}\nbest_score={best_score:.12f}\n")

        for (n_value, r_value) in candidates:
            now = datetime.now()
            if now - start_time > timeout:
                print("\nTIMEOUT")
                f.write("\nTIMEOUT\n")
                break

            completed_runs += 1
            copy(best_path, "submission.csv")

            print(f"\n[{completed_runs}] run bbox3 -n {n_value} -r {r_value}")
            f.write(f"\n[{now}] run bbox3 -n {n_value} -r {r_value}\n")

            try:
                result = subprocess.run(
                    ["./bbox3", "-n", str(n_value), "-r", str(r_value)],
                    capture_output=True,
                    text=True,
                    timeout=1200
                )
                if result.stdout:
                    f.write(result.stdout + "\n")
                if result.stderr:
                    f.write("STDERR:\n" + result.stderr + "\n")
            except subprocess.TimeoutExpired:
                print("bbox3 timeout")
                f.write("bbox3 timeout\n")
                continue
            except Exception as e:
                print("bbox3 error", e)
                f.write(f"bbox3 error {e}\n")
                continue

            fix_direction("submission.csv", "submission.csv")

            quick = score_and_validate_submission("submission.csv", max_n=200, check_overlaps=False)
            new_score = quick["total_score"]
            print(f"quick_score={new_score:.12f}  best={best_score:.12f}")
            f.write(f"quick_score={new_score:.12f}  best={best_score:.12f}\n")

            if new_score < best_score - 1e-9:
                full = score_and_validate_submission("submission.csv", max_n=200, check_overlaps=True)
                if full["status"] == "SUCCESS" and full["total_score"] < best_score - 1e-9:
                    best_score = full["total_score"]
                    copy("submission.csv", best_path)
                    stamp = f"bbox_sub/best_n{n_value}_r{r_value}_{completed_runs}.csv"
                    copy(best_path, stamp)
                    print(f"ACCEPT ✅ new best_score={best_score:.12f}")
                    f.write(f"ACCEPT new best_score={best_score:.12f} saved {stamp}\n")
                else:
                    print("REJECT (overlaps/full-worse)")
                    f.write("REJECT (overlaps/full-worse)\n")
                    copy(best_path, "submission.csv")
            else:
                copy(best_path, "submission.csv")

        copy(best_path, "submission.csv")
        end_time = datetime.now()
        f.write(f"\nEND {end_time}\nbest_score={best_score:.12f}\n{'='*60}\n")

    print(f"\nDONE. best_score={best_score:.12f}")

run_bbox_simple_with_timeout(debug=DEBUG)

In [ ]:
import pandas as pd
import numpy as np
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
from shapely.strtree import STRtree

getcontext().prec = 30
scale_factor = Decimal('1e20')


class ChristmasTree:
    """Represents a single, rotatable Christmas tree of a fixed size."""

    def __init__(self, center_x="0", center_y="0", angle="0"):
        """Initializes the Christmas tree with a specific position and rotation."""
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)

        trunk_w = Decimal("0.15")
        trunk_h = Decimal("0.2")
        base_w = Decimal("0.7")
        mid_w = Decimal("0.4")
        top_w = Decimal("0.25")
        tip_y = Decimal("0.8")
        tier_1_y = Decimal("0.5")
        tier_2_y = Decimal("0.25")
        base_y = Decimal("0.0")
        trunk_bottom_y = -trunk_h

        # Define the 15 vertices of the tree polygon
        initial_polygon = Polygon(
            [
                (Decimal("0.0") * scale_factor, tip_y * scale_factor),
                (top_w / Decimal("2") * scale_factor, tier_1_y * scale_factor),
                (top_w / Decimal("4") * scale_factor, tier_1_y * scale_factor),
                (mid_w / Decimal("2") * scale_factor, tier_2_y * scale_factor),
                (mid_w / Decimal("4") * scale_factor, tier_2_y * scale_factor),
                (base_w / Decimal("2") * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal("2") * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal("2") * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal("2")) * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal("2")) * scale_factor, base_y * scale_factor),
                (-(base_w / Decimal("2")) * scale_factor, base_y * scale_factor),
                (-(mid_w / Decimal("4")) * scale_factor, tier_2_y * scale_factor),
                (-(mid_w / Decimal("2")) * scale_factor, tier_2_y * scale_factor),
                (-(top_w / Decimal("4")) * scale_factor, tier_1_y * scale_factor),
                (-(top_w / Decimal("2")) * scale_factor, tier_1_y * scale_factor),
            ]
        )
        
        # Apply rotation and translation to the polygon
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(
            rotated, 
            xoff=float(self.center_x * scale_factor), 
            yoff=float(self.center_y * scale_factor)
        )


def load_configuration_from_df(n: int, df: pd.DataFrame) -> list[ChristmasTree]:
    """
    Loads all trees for a given N from the submission DataFrame.
    """
    group_data = df[df["id"].str.startswith(f"{n:03d}_")]
    trees = []
    for _, row in group_data.iterrows():
        # Remove 's' prefix and convert to string for Decimal constructor
        x = str(row["x"])[1:]
        y = str(row["y"])[1:]
        deg = str(row["deg"])[1:]
        
        # Ensure values are present before passing to ChristmasTree constructor
        if x and y and deg:
            trees.append(ChristmasTree(x, y, deg))
        else:
             # Handle cases where configuration might be incomplete/missing
             pass 
             
    return trees


def get_score(trees: list[ChristmasTree], n: int) -> float:
    """
    Calculates the score (S^2 / N) for a given configuration of trees.
    S is the side length of the minimum bounding square.
    """
    if not trees:
        return 0.0

    # Collect all exterior points from all tree polygons, scale them back down
    xys = np.concatenate([np.asarray(t.polygon.exterior.xy).T / float(scale_factor) for t in trees])
    
    min_x, min_y = xys.min(axis=0)
    max_x, max_y = xys.max(axis=0)
    
    side_length = max(max_x - min_x, max_y - min_y)
    
    # Score is S^2 / N
    score = side_length**2 / n
    return score

def has_overlap(trees: list[ChristmasTree]) -> bool:
    """Check if any two ChristmasTree polygons overlap."""
    if len(trees) <= 1:
        return False

    polygons = [t.polygon for t in trees]
    # Use STRtree for efficient proximity queries (optimizes checking pairs)
    tree_index = STRtree(polygons)

    for i, poly in enumerate(polygons):
        # Query for polygons whose bounding boxes overlap with poly
        # This returns the indices of potential overlaps
        indices = tree_index.query(poly)
        
        for idx in indices:
            # Skip checking the polygon against itself
            if idx == i:
                continue
                
            # Perform the precise intersection check
            if poly.intersects(polygons[idx]) and not poly.touches(polygons[idx]):
                # Overlap found!
                return True
    return False

# ----------------------------------------------------------------------


In [ ]:
# Example usage (assuming 'submission.csv' exists in the current directory)
result = score_and_validate_submission("submission.csv", max_n=200)
print(result)

In [ ]:
import csv


def load_groups(filename):
    """
    Загружает файл в словарь:
    {
        '001': [строка1, строка2],
        '002': [...],
        ...
    }
    """
    groups = {}
    with open(filename, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)  # сохраняем заголовок
        for row in reader:
            full_id = row[0]
            group = full_id.split('_')[0]

            groups.setdefault(group, []).append(row)

    return header, groups


def replace_group(target_file, donor_file, group_id, output_file=None):
    """
    target_file – файл, в котором меняем группу
    donor_file  – эталонный файл-источник
    group_id    – '004'
    output_file – куда сохранить (если None – перезапись target_file)
    """
    if output_file is None:
        output_file = target_file

    # Загружаем оба файла
    header_t, groups_t = load_groups(target_file)
    header_d, groups_d = load_groups(donor_file)

    # if header_t != header_d:
    #     raise ValueError("Ошибка: заголовки файлов отличаются!")

    if group_id not in groups_d:
        raise ValueError(f"В файле-донора нет группы {group_id}")

    # Заменяем
    groups_t[group_id] = groups_d[group_id]

    # Сохраняем результат
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header_t)

        # сортируем группы по номеру, чтобы порядок не сломать
        for g in sorted(groups_t.keys(), key=lambda x: int(x)):
            for row in groups_t[g]:
                writer.writerow(row)

    print(f"✔ Группа {group_id} заменена и сохранена в {output_file}")


GROIP_IDXS = result['failed_overlap_n']
if GROIP_IDXS:
    for GROIP_ID in GROIP_IDXS:
        replace_group(
            target_file="submission.csv",
            donor_file="/kaggle/input/santa-2025-csv/santa-2025.csv",
            group_id=f'{GROIP_ID:03d}',
            output_file="submission.csv"
        )

In [ ]:
from IPython.display import display, FileLink
from zipfile import ZipFile, ZIP_DEFLATED as ZD
from datetime import datetime
from glob import glob

files = glob(f'*.csv') + glob(f'*.log') + glob(f'bbox_sub/*.csv')
formatted_time = datetime.now().strftime("%Y-%m-%d-%H-%M")
zip_filename = f'kaggle_bbox_{formatted_time}.zip'
with ZipFile(zip_filename, 'w',  compression=ZD, compresslevel=9) as zip_file:
    for filename in files:
        print(filename)
        zip_file.write(filename)
FileLink(zip_filename)